# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

In [1]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta_vals = np.linspace(-6, 6, 500)

curves = [
    (0.5, -2, "dash"),
    (0.5, 0, "dash"),
    (0.5, 2, "dash"),
    (1.5, -2, "solid"),
    (1.5, 0, "solid"),
    (1.5, 2, "solid")
]

fig1 = go.Figure()

for a, b, style in curves:
    fig1.add_trace(
        go.Scatter(
            x=theta_vals,
            y=p_i(theta_vals, a, b),
            mode="lines",
            name=f"a = {a}, b = {b}",
            line=dict(width=2.5, dash=style)
        )
    )

fig1.update_layout(
    title="Two-Parameter Logistic (2PL) Item Response Curves",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="P(Y_i = 1 | θ)",
    xaxis=dict(range=[-6, 6]),
    yaxis=dict(range=[0, 1.05]),
    template="plotly_white",
    legend=dict(x=0.02, y=0.98)
)

fig1.show()

print("Sequential likelihood for one response:")
print("L(y_k | θ) = p_k(θ)^y_k (1 - p_k(θ))^(1 - y_k)")
print()
print("Joint likelihood for the running history y^(k):")
print("L(y^(k) | θ) = ∏_{i=1}^k p_i(θ)^(y_i) (1 - p_i(θ))^(1 - y_i)")
print()
print("Recursive posterior update:")
print("f(θ | y^(k)) ∝ [p_k(θ)^y_k (1 - p_k(θ))^(1 - y_k)] f(θ | y^(k-1))")
print()
print("If y_k = 1 for a difficult item with large b_k, the likelihood term favors larger θ, so the posterior peak shifts to the right.")
print("A larger discrimination a_k makes the likelihood steeper and the posterior sharper; a small a_k makes the update flatter and less informative.")
print("Numerically, evaluate the posterior on a θ-grid, multiply by the likelihood at each step, then normalize using the trapezoidal rule.")

np.random.seed(42)

theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1201)

current_posterior = stats.norm.pdf(theta_grid, 0, 1)
current_posterior /= np.trapz(current_posterior, theta_grid)

running_bayes = [np.trapz(theta_grid * current_posterior, theta_grid)]
running_map = [theta_grid[np.argmax(current_posterior)]]
steps = [0]

item_a = []
item_b = []
responses = []

for k in range(n_items):
    a_k = np.random.uniform(0.5, 2.0)
    b_k = np.random.normal(0, 1)
    prob_true = p_i(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < prob_true else 0

    prob_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (prob_grid ** y_k) * ((1 - prob_grid) ** (1 - y_k))

    current_posterior = current_posterior * likelihood
    current_posterior /= np.trapz(current_posterior, theta_grid)

    bayes_k = np.trapz(theta_grid * current_posterior, theta_grid)
    map_k = theta_grid[np.argmax(current_posterior)]

    running_bayes.append(bayes_k)
    running_map.append(map_k)
    steps.append(k + 1)

    item_a.append(a_k)
    item_b.append(b_k)
    responses.append(y_k)

fig2 = go.Figure()

fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    line_width=2,
    annotation_text="True ability θ = 0.75",
    annotation_position="bottom right"
)

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=running_bayes,
        mode="lines+markers",
        name="Posterior Mean",
        line=dict(width=2.5),
        marker=dict(size=6)
    )
)

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=running_map,
        mode="lines+markers",
        name="MAP Estimate",
        line=dict(width=2.5),
        marker=dict(size=6, symbol="square")
    )
)

fig2.update_layout(
    title="Running Estimators of Latent Ability Over 20 Items",
    xaxis_title="Item Index k",
    yaxis_title="Estimated Ability θ̂",
    xaxis=dict(tickmode="linear", tick0=0, dtick=2),
    template="plotly_white",
    legend=dict(x=0.02, y=0.98)
)

fig2.show()

print()
print("Final simulation summary")
print(f"True ability: {theta_true:.2f}")
print(f"Posterior mean after 20 items: {running_bayes[-1]:.4f}")
print(f"MAP after 20 items: {running_map[-1]:.4f}")
print()
print("As k increases, both estimators usually move closer to the true ability and become more stable.")
print("That shrinking gap means the platform is learning the user's ability and gaining confidence from the response history.")

Sequential likelihood for one response:
L(y_k | θ) = p_k(θ)^y_k (1 - p_k(θ))^(1 - y_k)

Joint likelihood for the running history y^(k):
L(y^(k) | θ) = ∏_{i=1}^k p_i(θ)^(y_i) (1 - p_i(θ))^(1 - y_i)

Recursive posterior update:
f(θ | y^(k)) ∝ [p_k(θ)^y_k (1 - p_k(θ))^(1 - y_k)] f(θ | y^(k-1))

If y_k = 1 for a difficult item with large b_k, the likelihood term favors larger θ, so the posterior peak shifts to the right.
A larger discrimination a_k makes the likelihood steeper and the posterior sharper; a small a_k makes the update flatter and less informative.
Numerically, evaluate the posterior on a θ-grid, multiply by the likelihood at each step, then normalize using the trapezoidal rule.


/tmp/ipykernel_1186/1461306177.py:64: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_1186/1461306177.py:66: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_1186/1461306177.py:84: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_1186/1461306177.py:86: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.




Final simulation summary
True ability: 0.75
Posterior mean after 20 items: 0.3499
MAP after 20 items: 0.3500

As k increases, both estimators usually move closer to the true ability and become more stable.
That shrinking gap means the platform is learning the user's ability and gaining confidence from the response history.


# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

In [2]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(42)

def beta_map(a, b):
    if a > 1 and b > 1:
        return (a - 1) / (a + b - 2)
    if a <= 1 and b > 1:
        return 0.0
    if a > 1 and b <= 1:
        return 1.0
    return np.nan

theta_grid = np.linspace(0, 1, 500)

beta_configs = [
    (1, 1, "Beta(1,1)  Uniform prior", "gray", "dash"),
    (2, 8, "Beta(2,8)  Right-skewed", "blue", "solid"),
    (8, 2, "Beta(8,2)  Left-skewed", "green", "solid"),
]

fig1 = go.Figure()

for a, b, name, color, dash in beta_configs:
    fig1.add_trace(
        go.Scatter(
            x=theta_grid,
            y=stats.beta.pdf(theta_grid, a, b),
            mode="lines",
            name=name,
            line=dict(color=color, dash=dash, width=2.5),
        )
    )

fig1.update_layout(
    title="Beta Distribution PDFs",
    xaxis_title="θ",
    yaxis_title="Density",
    xaxis=dict(range=[0, 1]),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(x=0.02, y=0.98),
)

fig1.show()

print("1) Single-response likelihood:")
print("L(y_k | θ) = θ^y_k (1 - θ)^(1 - y_k)")
print()
print("Joint likelihood for the running history y^(k):")
print("L(y^(k) | θ) = ∏_{i=1}^k θ^y_i (1 - θ)^(1 - y_i)")
print()
print("2) Sequential Bayes update:")
print("f(θ | y^(k)) ∝ [θ^y_k (1 - θ)^(1 - y_k)] f(θ | y^(k-1))")
print()
print("3) Conjugate Beta update:")
print("If f(θ | y^(k-1)) = Beta(α_{k-1}, β_{k-1}), then")
print("α_k = α_{k-1} + y_k")
print("β_k = β_{k-1} + (1 - y_k)")
print("So f(θ | y^(k)) = Beta(α_k, β_k)")
print()
print("Posterior mean:")
print("E[θ | y^(k)] = α_k / (α_k + β_k)")
print()
print("4) Shift behaviour:")
print("If y_k = 1, α_k increases, so the posterior shifts right.")
print("If y_k = 0, β_k increases, so the posterior shifts left.")
print("This is closed-form, unlike non-conjugate models such as 2PL IRT, which need numerical approximation.")
print()
print("5) Point estimates:")
print("Bayes estimate = α_k / (α_k + β_k)")
print("MAP = (α_k - 1) / (α_k + β_k - 2) when α_k > 1 and β_k > 1")
print("For boundary cases, the MAP is at 0 or 1; for Beta(1,1), it is not unique.")
print()

theta_true = 0.35
n_impressions = 100
alpha0 = 1
beta0 = 1

alpha = alpha0
beta = beta0

steps = [0]
bayes_track = [alpha / (alpha + beta)]
map_track = [np.nan if alpha == 1 and beta == 1 else beta_map(alpha, beta)]

posterior_curves = []

for k in range(1, n_impressions + 1):
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    alpha += y_k
    beta += 1 - y_k
    steps.append(k)
    bayes_track.append(alpha / (alpha + beta))
    map_track.append(beta_map(alpha, beta))
    if k in [1, 2, 5, 10, 20, 50, 100]:
        posterior_curves.append((k, alpha, beta, y_k))

fig2 = go.Figure()

fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    line_width=2,
    annotation_text="True CTR = 0.35",
    annotation_position="bottom right",
)

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=bayes_track,
        mode="lines+markers",
        name="Posterior Mean",
        line=dict(width=2.5),
        marker=dict(size=6),
    )
)

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=map_track,
        mode="lines+markers",
        name="MAP Estimate",
        line=dict(width=2.5, dash="dot"),
        marker=dict(size=6, symbol="square"),
    )
)

fig2.update_layout(
    title="Sequential Beta-Binomial Estimation of CTR",
    xaxis_title="Impression k",
    yaxis_title="Estimated CTR",
    xaxis=dict(tickmode="linear", tick0=0, dtick=10),
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(x=0.02, y=0.98),
)

fig2.show()

print("6) Final sequential tracking summary")
print(f"Final α = {alpha}")
print(f"Final β = {beta}")
print(f"Final posterior mean = {bayes_track[-1]:.4f}")
print(f"Final MAP = {map_track[-1] if not np.isnan(map_track[-1]) else 'undefined at Beta(1,1) start only'}")
print()
print("As k approaches 100, both estimators usually move closer to the true CTR and become steadier.")
print("That means the evidence from the impressions is gradually dominating the initial prior.")

1) Single-response likelihood:
L(y_k | θ) = θ^y_k (1 - θ)^(1 - y_k)

Joint likelihood for the running history y^(k):
L(y^(k) | θ) = ∏_{i=1}^k θ^y_i (1 - θ)^(1 - y_i)

2) Sequential Bayes update:
f(θ | y^(k)) ∝ [θ^y_k (1 - θ)^(1 - y_k)] f(θ | y^(k-1))

3) Conjugate Beta update:
If f(θ | y^(k-1)) = Beta(α_{k-1}, β_{k-1}), then
α_k = α_{k-1} + y_k
β_k = β_{k-1} + (1 - y_k)
So f(θ | y^(k)) = Beta(α_k, β_k)

Posterior mean:
E[θ | y^(k)] = α_k / (α_k + β_k)

4) Shift behaviour:
If y_k = 1, α_k increases, so the posterior shifts right.
If y_k = 0, β_k increases, so the posterior shifts left.
This is closed-form, unlike non-conjugate models such as 2PL IRT, which need numerical approximation.

5) Point estimates:
Bayes estimate = α_k / (α_k + β_k)
MAP = (α_k - 1) / (α_k + β_k - 2) when α_k > 1 and β_k > 1
For boundary cases, the MAP is at 0 or 1; for Beta(1,1), it is not unique.



6) Final sequential tracking summary
Final α = 42
Final β = 60
Final posterior mean = 0.4118
Final MAP = 0.41

As k approaches 100, both estimators usually move closer to the true CTR and become steadier.
That means the evidence from the impressions is gradually dominating the initial prior.


# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In [3]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(24)

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n = 15

theta_grid = np.linspace(0.01, 1.0, 700)

alpha0 = 8
beta0 = 1.5

prior = stats.beta.pdf(theta_grid, alpha0, beta0)
prior /= np.trapezoid(prior, theta_grid)

posterior = prior.copy()

bayes_track = [np.trapezoid(theta_grid * posterior, theta_grid)]
map_track = [theta_grid[np.argmax(posterior)]]
sd_track = [np.sqrt(np.trapezoid((theta_grid - bayes_track[0]) ** 2 * posterior, theta_grid))]
steps = [0]

milestones = [0, 1, 2, 5, 10, 15]
posterior_at_milestones = {0: posterior.copy()}
observed_y = []

print("Prior belief boundaries")
print(f"E[Theta^(0)] = {alpha0 / (alpha0 + beta0):.6f}")
print("A Beta(8, 1.5) prior is appropriate because it places most probability mass near 1,")
print("so it reflects an initially optimistic belief that the component is likely healthy.")
print()

print("Structural likelihood")
print("L(y_k | theta) = [1 / (y_k * sigma * sqrt(2*pi))] * exp(-(ln(y_k) - ln(theta * K_nominal))^2 / (2*sigma^2)),  y_k > 0")
print("L(y^(k) | theta) = product over i=1 to k of L(y_i | theta)")
print()

print("Non-conjugate recursive posterior update")
print("f(theta | y^(k)) proportional to L(y_k | theta) * f(theta | y^(k-1))")
print()

print("Running point estimators")
print("theta_Bayes^(k) = integral over (0,1] of theta * f(theta | y^(k)) d theta")
print("theta_MAP^(k) = argmax over theta in (0,1] of f(theta | y^(k))")
print()

fig1 = go.Figure()

fig1.add_trace(
    go.Scatter(
        x=theta_grid,
        y=posterior,
        mode="lines",
        name="k = 0 prior Beta(8,1.5)",
        line=dict(width=2.5, dash="dash")
    )
)

for k in range(1, n + 1):
    eps = np.random.normal(0, sigma)
    y_k = theta_true * K_nominal * np.exp(eps)
    observed_y.append(y_k)

    likelihood = (
        (1.0 / (y_k * sigma * np.sqrt(2 * np.pi)))
        * np.exp(-((np.log(y_k) - np.log(theta_grid * K_nominal)) ** 2) / (2 * sigma ** 2))
    )

    posterior = posterior * likelihood
    posterior /= np.trapezoid(posterior, theta_grid)

    bayes_k = np.trapezoid(theta_grid * posterior, theta_grid)
    map_k = theta_grid[np.argmax(posterior)]
    sd_k = np.sqrt(np.trapezoid(((theta_grid - bayes_k) ** 2) * posterior, theta_grid))

    steps.append(k)
    bayes_track.append(bayes_k)
    map_track.append(map_k)
    sd_track.append(sd_k)

    if k in milestones:
        posterior_at_milestones[k] = posterior.copy()

for k in milestones[1:]:
    fig1.add_trace(
        go.Scatter(
            x=theta_grid,
            y=posterior_at_milestones[k],
            mode="lines",
            name=f"k = {k}",
            line=dict(width=2)
        )
    )

fig1.add_vline(
    x=theta_true,
    line_dash="dot",
    line_width=2,
    annotation_text="True stiffness efficiency = 0.68",
    annotation_position="top left"
)

fig1.update_layout(
    title="Sequential Bayesian SHM Update with Bounded Beta Prior and Log-Normal Likelihood",
    xaxis_title="Remaining stiffness efficiency θ",
    yaxis_title="Posterior density",
    xaxis=dict(range=[0.01, 1.0]),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(x=0.02, y=0.98)
)

fig1.show()

fig2 = go.Figure()

fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    line_width=2,
    annotation_text="True θ = 0.68",
    annotation_position="bottom right"
)

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=bayes_track,
        mode="lines+markers",
        name="Posterior mean",
        line=dict(width=2.5),
        marker=dict(size=6)
    )
)

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=map_track,
        mode="lines+markers",
        name="MAP estimate",
        line=dict(width=2.5, dash="dot"),
        marker=dict(size=6, symbol="square")
    )
)

fig2.update_layout(
    title="Convergence of Running Estimators for Structural Health Monitoring",
    xaxis_title="Inspection step k",
    yaxis_title="Estimated θ",
    xaxis=dict(tickmode="linear", tick0=0, dtick=1),
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(x=0.02, y=0.98)
)

fig2.show()

tol = 0.05
close_steps = [
    k for k, (b, m) in enumerate(zip(bayes_track, map_track))
    if np.isfinite(m) and abs(b - theta_true) <= tol and abs(m - theta_true) <= tol
]

print()
if close_steps:
    k_star = close_steps[0]
    print(f"Using a ±{tol:.2f} band, both estimators first become close to the true damage state at step k = {k_star}.")
else:
    print(f"Using a ±{tol:.2f} band, both estimators did not simultaneously enter the target band within 15 readings.")

print(f"Final posterior mean after {n} readings: {bayes_track[-1]:.6f}")
print(f"Final MAP after {n} readings: {map_track[-1]:.6f}")
print(f"Final posterior spread (SD): {sd_track[-1]:.6f}")
print()
print("Interpretation:")
print("As more sensor readings arrive, the posterior density becomes narrower and shifts toward the true state.")
print("That narrowing means the monitoring system is gaining confidence and the optimistic prior is being overridden by data.")
print("In engineering terms, a tighter posterior around 0.68 suggests the damage level is being isolated more reliably, which helps compare the structure against safety thresholds.")

Prior belief boundaries
E[Theta^(0)] = 0.842105
A Beta(8, 1.5) prior is appropriate because it places most probability mass near 1,
so it reflects an initially optimistic belief that the component is likely healthy.

Structural likelihood
L(y_k | theta) = [1 / (y_k * sigma * sqrt(2*pi))] * exp(-(ln(y_k) - ln(theta * K_nominal))^2 / (2*sigma^2)),  y_k > 0
L(y^(k) | theta) = product over i=1 to k of L(y_i | theta)

Non-conjugate recursive posterior update
f(theta | y^(k)) proportional to L(y_k | theta) * f(theta | y^(k-1))

Running point estimators
theta_Bayes^(k) = integral over (0,1] of theta * f(theta | y^(k)) d theta
theta_MAP^(k) = argmax over theta in (0,1] of f(theta | y^(k))




Using a ±0.05 band, both estimators first become close to the true damage state at step k = 3.
Final posterior mean after 15 readings: 0.683132
Final MAP after 15 readings: 0.681330
Final posterior spread (SD): 0.026398

Interpretation:
As more sensor readings arrive, the posterior density becomes narrower and shifts toward the true state.
That narrowing means the monitoring system is gaining confidence and the optimistic prior is being overridden by data.
In engineering terms, a tighter posterior around 0.68 suggests the damage level is being isolated more reliably, which helps compare the structure against safety thresholds.


# Q. Gaussian Mixture Clustering as Conditional Updating

In [10]:
from IPython.display import display, Markdown

display(Markdown(r"""
# Gaussian Mixture Models as Conditional Expectation Models

## Part 1 – Marginal Density and Posterior Cluster Probabilities

# ---

# 1. Deriving the Marginal Density

Consider a dataset

$$
x_1,x_2,\dots,x_n\in\mathbb{R}^d.
$$

A Gaussian Mixture Model (GMM) assumes that every observation is generated from one of $K$ latent Gaussian components. The hidden cluster membership of observation $x_i$ is represented by the random variable

$$
C_i\in\{1,2,\ldots,K\},
$$

with prior probability

$$
P(C_i=k)=\phi_k,
$$

where

$$
\phi_k\ge0,
\qquad
\sum_{k=1}^{K}\phi_k=1.
$$

Conditional on belonging to cluster $k$, the observation follows a multivariate Gaussian distribution,

$$
X_i\mid C_i=k
\sim
\mathcal{N}(\mu_k,\Sigma_k).
$$

To determine the unconditional (marginal) density of an observation, we apply the **Law of Total Probability** by summing over every possible cluster assignment.

The joint probability of observing $x_i$ and belonging to cluster $k$ is

$$
P(X_i=x_i,C_i=k)
=
P(X_i=x_i\mid C_i=k)\,P(C_i=k).
$$

Therefore,

$$
p(x_i)
=
\sum_{k=1}^{K}
P(X_i=x_i,C_i=k).
$$

Substituting the Gaussian likelihood and the prior mixture probabilities gives

$$
p(x_i)
=
\sum_{k=1}^{K}
P(X_i=x_i\mid C_i=k)\,P(C_i=k)
=
\sum_{k=1}^{K}
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k).
$$

Hence, the marginal density of an observation is

$$
\boxed{
p(x_i)
=
\sum_{k=1}^{K}
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}
$$

# ---

### Why is this called a Gaussian Mixture Density?

The resulting probability density is called a **Gaussian mixture density** because it is constructed as a weighted combination of several Gaussian component distributions.

Each Gaussian distribution

$$
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
$$

models one underlying cluster in the dataset, while the mixture weight

$$
\phi_k
$$

represents the prior probability that an arbitrary observation originates from cluster $k$.

The complete density therefore combines multiple Gaussian distributions into a single probability model,

$$
p(x_i)
=
\phi_1\mathcal{N}_1
+
\phi_2\mathcal{N}_2
+\cdots+
\phi_K\mathcal{N}_K.
$$

Unlike a single Gaussian distribution, which can only represent one approximately elliptical cluster, a Gaussian mixture density can approximate highly irregular, skewed, or multimodal distributions by combining several Gaussian components together.

# ---

# 2. Deriving the Posterior Cluster Probability

After observing a particular data point $x_i$, we wish to determine the probability that it originated from each cluster.

Using **Bayes' Theorem**,

$$
P(C_i=k\mid X_i=x_i)
=
\frac{
P(X_i=x_i\mid C_i=k)\,P(C_i=k)
}{
P(X_i=x_i)
}.
$$

From Part 1,

$$
P(X_i=x_i)
=
\sum_{j=1}^{K}
\phi_j
\mathcal{N}(x_i\mid\mu_j,\Sigma_j).
$$

Substituting the likelihood and prior gives

$$
P(C_i=k\mid X_i=x_i)
=
\frac{
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}{
\sum_{j=1}^{K}
\phi_j
\mathcal{N}(x_i\mid\mu_j,\Sigma_j)
}.
$$

The posterior probability is therefore

$$
\boxed{
P(C_i=k\mid X_i=x_i)
=
\frac{
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}{
\sum_{j=1}^{K}
\phi_j
\mathcal{N}(x_i\mid\mu_j,\Sigma_j)
}
}
$$

This quantity is called the **responsibility** of cluster $k$ for observation $x_i$ and is denoted by

$$
\boxed{
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i)
}.
$$

# ---

### Interpretation of the Responsibility

The responsibility

$$
\gamma_{ik}
$$

is the **posterior probability** that observation $x_i$ belongs to cluster $k$ after incorporating the observed data.

Its interpretation follows directly from Bayes' theorem:

- **Prior Information:** Before observing the data, the probability of belonging to cluster $k$ is represented by the mixture weight

  $$
  \phi_k.
  $$

- **Likelihood:** The Gaussian density

  $$
  \mathcal{N}(x_i\mid\mu_k,\Sigma_k)
  $$

  measures how compatible the observed point is with cluster $k$.

- **Posterior Update:** Bayes' theorem combines these two sources of information to produce the updated probability

  $$
  \gamma_{ik}.
  $$

The responsibilities satisfy

$$
0\le\gamma_{ik}\le1,
$$

and

$$
\sum_{k=1}^{K}\gamma_{ik}=1,
$$

which means every observation distributes one unit of probability across all clusters.

Observations located near the overlap of two Gaussian components may have responsibilities such as

$$
(\gamma_{i1},\gamma_{i2},\gamma_{i3})
=
(0.45,\;0.50,\;0.05),
$$

indicating uncertainty in cluster membership.

Conversely, a point lying very close to the center of one cluster may have responsibilities such as

$$
(0.98,\;0.01,\;0.01),
$$

showing that nearly all posterior probability is assigned to a single cluster.

Thus, the responsibility provides a **soft probabilistic assignment** instead of making an immediate hard decision about cluster membership.
"""))
# =============================================================================

# =============================================================================

display(Markdown(r"""

# Part 2

---

# 4. From Soft Assignment to Hard Clustering

In a Gaussian Mixture Model (GMM), the conditional expectation vector

$$
\mathbb{E}[Z_i \mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}
$$

provides a **soft assignment** of observation $x_i$ across all clusters.

Each responsibility satisfies

$$
0 \le \gamma_{ik}\le 1,
\qquad
\sum_{k=1}^{K}\gamma_{ik}=1,
$$

meaning every observation belongs to every cluster with different posterior
probabilities.

The corresponding **hard assignment** is obtained by selecting the cluster with
the highest posterior probability:

$$
\widehat C_i
=
\operatorname*{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$

---

### Soft Clustering

Soft clustering preserves uncertainty in the cluster assignment.

Instead of forcing a point into a single cluster, the model produces an entire
probability vector describing how strongly the observation belongs to every
component.

For example,

$$
\mathbb{E}[Z_i\mid X_i=x_i]
=
\begin{bmatrix}
0.15\\
0.70\\
0.15
\end{bmatrix}
$$

indicates that the observation most likely belongs to Cluster 2 while still
retaining some probability of belonging to Clusters 1 and 3.

This probabilistic representation is especially useful when observations lie
near overlapping cluster boundaries.

---

### Hard Clustering

Hard clustering converts the posterior probability vector into a single class
label.

Using the previous example,

$$
\widehat C_i
=
\operatorname*{arg\,max}
(0.15,\;0.70,\;0.15)
=
2.
$$

The observation is assigned entirely to Cluster 2, while the remaining
probabilities are discarded.

---

### Comparison

| Soft Clustering | Hard Clustering |
|----------------- |-----------------|
| Produces posterior probabilities | Produces one cluster label |
| Preserves uncertainty | Removes uncertainty |
| Observation belongs partially to every cluster | Observation belongs to exactly one cluster |
| Used during EM parameter estimation | Used for final classification and visualization |

---

### Conclusion

The conditional expectation

$$
\mathbb{E}[Z_i\mid X_i=x_i]
$$

contains the complete probabilistic description of cluster membership.

The hard assignment

$$
\widehat C_i
=
\operatorname*{arg\,max}_{k}
\gamma_{ik}
$$

is simply a deterministic decision obtained from this probability vector.

---

# 5. Conditional Expectation of the Observation Given the Cluster

By definition of the Gaussian Mixture Model,

$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k).
$$

The expected value of a multivariate Gaussian distribution equals its mean
vector.

Therefore,

$$
\boxed{
\mathbb E[X_i\mid C_i=k]=\mu_k
}
$$

or equivalently,

$$
\mathbb E[X_i\mid C_i=k]
=
\int_{\mathbb R^d}
x\,
\mathscr N(x\mid\mu_k,\Sigma_k)
dx
=
\mu_k.
$$

---

## Why is $\mu_k$ the Center of Cluster $k$?

The vector $\mu_k$ represents the centroid of the Gaussian component because

* it is the expected location of observations generated from cluster $k$,

* it is the point where the Gaussian density reaches its maximum,

* the Gaussian distribution is symmetric around $\mu_k$,

* every direction away from $\mu_k$ balances equally in expectation.

Hence,

$$
\boxed{
\mu_k
=
\text{Center of Cluster }k
}
$$

---

## Comparing the Two Conditional Expectations

### (a)

$$
\mathbb E[Z_i\mid X_i=x_i]
$$

This quantity conditions on an observed data point.

The location of the point is fixed, while the cluster identity is unknown.

Its output is

$$
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix},
$$

which represents the posterior probability that the observation belongs to each
cluster.

Therefore,

$$
\boxed{
\mathbb E[Z_i\mid X_i=x_i]
=
\text{Soft cluster membership vector.}
}
$$

---

### (b)

$$
\mathbb E[X_i\mid C_i=k]
$$

This quantity conditions on the cluster identity.

The cluster is fixed, while the observation is random.

Its output is

$$
\mu_k,
$$

the average location of observations generated from that cluster.

Therefore,

$$
\boxed{
\mathbb E[X_i\mid C_i=k]
=
\text{Center of Cluster }k.
}
$$

---

### Summary Comparison

| Quantity | Conditions On | Unknown Variable | Output | Interpretation |
|-----------|---------------|-----------------|--------|----------------|
| $\mathbb E[Z_i\mid X_i=x_i]$ | Observation | Cluster identity | Probability vector | Soft cluster membership |
| $\mathbb E[X_i\mid C_i=k]$ | Cluster | Observation | Mean vector | Cluster center |

---

### Conclusion

Although both expressions are conditional expectations, they answer completely
different questions.

The expectation

$$
\mathbb E[Z_i\mid X_i=x_i]
$$

estimates **which cluster generated an observed point**, whereas

$$
\mathbb E[X_i\mid C_i=k]
$$

estimates **where observations from a particular cluster are expected to be
located**.

---

# 6. The Complete-Data Likelihood

Suppose that the latent indicator variables

$$
z_{ik}
\in
\{0,1\}
$$

are observed.

The complete-data likelihood is

$$
p(x_1,\ldots,x_n,z_1,\ldots,z_n)
=
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathscr N(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$

---

## Taking the Logarithm

Applying the logarithm,

$$
\ell_c
=
\log
\left(
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathscr N(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}
\right).
$$

Using

$$
\log\left(\prod a_i\right)
=
\sum\log(a_i),
$$

gives

$$
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\log
\left(
\left[
\phi_k
\mathscr N(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}
\right).
$$

Applying

$$
\log(a^b)=b\log(a),
$$

yields

$$
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\log
\left(
\phi_k
\mathscr N(x_i\mid\mu_k,\Sigma_k)
\right).
$$

Finally, using

$$
\log(ab)
=
\log a+\log b,
$$

we obtain

$$
\boxed{
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
+
\log
\mathscr N(x_i\mid\mu_k,\Sigma_k)
\right]
}
$$

which is the required complete-data log-likelihood.

---

## Why is this Easy to Maximize?

If every indicator variable $z_{ik}$ were known, then the assignment of every
observation to a cluster would already be available.

Consequently,

* each Gaussian component could be estimated independently,

* there would be no logarithm of a summation,

* observations assigned to each cluster could be treated as belonging to a
single Gaussian distribution,

* the mixture weights would simply equal the proportion of observations assigned
to each cluster.

The maximum likelihood estimates become

$$
\phi_k
=
\frac{\text{Number of observations in Cluster }k}{n},
$$

$$
\mu_k
=
\text{Sample mean of Cluster }k,
$$

and

$$
\Sigma_k
=
\text{Sample covariance of Cluster }k.
$$

---

### Conclusion

The complete-data log-likelihood is straightforward to optimize because the
hidden variables separate the observations into independent Gaussian
components.

The difficulty of Gaussian Mixture Models arises precisely because these latent
variables are unknown, motivating the Expectation-Maximization (EM) algorithm,
which replaces the unknown indicators with their conditional expectations.

"""))

display(Markdown(r"""

# Part 6. The Complete-Data Likelihood

## 1. Derivation of the Complete-Data Log-Likelihood

If the latent cluster indicators are observed, the complete-data likelihood is

$$
p(x_1,\ldots,x_n,z_1,\ldots,z_n)
=
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$

Taking the natural logarithm,

$$
\ell_c
=
\log
\left(
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}
\right).
$$

Using the logarithm property

$$
\log\left(\prod a_i\right)=\sum\log(a_i),
$$

gives

$$
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\log
\left(
\left[
\phi_k
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}
\right).
$$

Applying

$$
\log(a^b)=b\log(a),
$$

we obtain

$$
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\log
\left(
\phi_k
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right).
$$

Finally, using

$$
\log(ab)=\log a+\log b,
$$

the complete-data log-likelihood becomes

$$
\boxed{
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
+
\log
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right]
}
$$

which is the required expression.

---

## 2. Why the Complete-Data Log-Likelihood is Easy to Maximize

If every latent indicator $z_{ik}$ were known, the optimization problem would become much simpler.

Each observation would already be assigned to a particular Gaussian component, so there would be no uncertainty regarding cluster membership.

The logarithm transforms the product of probabilities into a summation, causing the contribution of every Gaussian component to become additive rather than coupled through a logarithm of sums.

Consequently,

* the mixture weights $\phi_k$ can be estimated directly from the proportion of observations assigned to each cluster,

* the mean vector $\mu_k$ becomes the sample average of the observations assigned to cluster $k$, and

* the covariance matrix $\Sigma_k$ becomes the sample covariance of those observations.

Thus, if the values of $z_{ik}$ were known, the difficult mixture optimization problem decomposes into $K$ independent maximum-likelihood estimation problems for ordinary multivariate Gaussian distributions.

---

# Part 7. The EM Interpretation

## 1. Expected Complete-Data Log-Likelihood

In practice, the latent indicators $z_{ik}$ are never observed.

Instead, the Expectation-Maximization (EM) algorithm replaces every unknown indicator by its conditional expectation given the observed data and the current parameter estimates.

Beginning with the complete-data log-likelihood,

$$
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
+
\log
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right],
$$

we take the conditional expectation with respect to the posterior distribution of the latent variables:

$$
Q
=
\mathbb{E}
\left[
\ell_c
\mid
X=x
\right].
$$

Substituting the expression for $\ell_c$,

$$
Q
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\mathbb{E}
[
Z_{ik}
\mid
X_i=x_i
]
\left[
\log\phi_k
+
\log
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right].
$$

From Part 3,

$$
\mathbb{E}
[
Z_{ik}
\mid
X_i=x_i
]
=
\gamma_{ik},
$$

where

$$
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i)
$$

is the posterior responsibility.

Therefore,

$$
\boxed{
Q
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\gamma_{ik}
\left[
\log\phi_k
+
\log
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right]
}
$$

which is called the expected complete-data log-likelihood or the Q-function.

---

## 2. Interpretation of the E-Step

The Expectation step performs a conditional update of the latent cluster memberships.

Instead of assigning each observation permanently to one cluster, the algorithm computes the posterior probability that every observation belongs to every Gaussian component.

Each responsibility

$$
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i)
$$

combines

* the prior probability of cluster $k$,

$$
\phi_k,
$$

and

* the likelihood that cluster $k$ generated the observation,

$$
\mathscr{N}(x_i\mid\mu_k,\Sigma_k),
$$

through Bayes' theorem.

Consequently, the E-step transforms the unknown binary indicator variables into continuous probabilities representing updated beliefs about cluster membership.

These posterior probabilities become the weights used during the Maximization step to update the model parameters.

---

# Part 8. Parameter Updates

After computing the posterior responsibilities during the E-step, the Maximization step updates every model parameter by maximizing the expected complete-data log-likelihood.

## Effective Number of Observations

The effective number of observations assigned to cluster $k$ is

$$
\boxed{
N_k
=
\sum_{i=1}^{n}
\gamma_{ik}
}
$$

which represents the total posterior membership weight of cluster $k$.

---

## Updating the Mixture Weights

The updated prior probability of cluster $k$ is

$$
\boxed{
\phi_k^{\mathrm{new}}
=
\frac{N_k}{n}
}
$$

ensuring

$$
\sum_{k=1}^{K}
\phi_k^{\mathrm{new}}
=
1.
$$

---

## Updating the Cluster Means

The updated mean vector is

$$
\boxed{
\mu_k^{\mathrm{new}}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}x_i
}
$$

which is a weighted average of all observations.

Observations with larger posterior probabilities contribute more heavily to the cluster center.

---

## Updating the Covariance Matrices

The updated covariance matrix is

$$
\boxed{
\Sigma_k^{\mathrm{new}}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}
(x_i-\mu_k^{\mathrm{new}})
(x_i-\mu_k^{\mathrm{new}})^T
}
$$

which measures the weighted spread of the observations around the updated mean.

---

## Interpretation of the Responsibilities

The responsibility

$$
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i)
$$

acts as a fractional membership weight rather than a binary assignment.

If

$$
\gamma_{ik}=1,
$$

the observation belongs completely to cluster $k$.

If

$$
\gamma_{ik}=0,
$$

the observation contributes nothing to that cluster.

Intermediate values indicate partial membership, allowing observations located between overlapping Gaussian components to influence multiple clusters simultaneously.

This weighted updating mechanism enables the Gaussian Mixture Model to represent uncertainty in cluster membership while continuously refining the model parameters throughout the EM iterations.

"""))

import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

file_path = ""

# Removed the problematic kagglehub.load_dataset call
# df = kagglehub.load_dataset(
#     KaggleDatasetAdapter.PANDAS,
#     "arjunbhasin2013/ccdata",
#     file_path
# )

dataset_path = kagglehub.dataset_download("arjunbhasin2013/ccdata")
print("Dataset Path:", dataset_path)

print("Files:")
print(os.listdir(dataset_path))

df = pd.read_csv(os.path.join(dataset_path, "CC GENERAL.csv"))


class GMMFinancialSegmenter:

    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.model = GaussianMixture(
            n_components=n_components,
            covariance_type="full",
            random_state=random_state
        )

    def prepare_data(self, dataframe, feature_columns, test_size=0.2):
        X = dataframe[feature_columns].dropna().values
        X_scaled = self.scaler.fit_transform(X)

        X_train, X_test = train_test_split(
            X_scaled,
            test_size=test_size,
            random_state=self.random_state
        )

        return X_train, X_test

    def fit(self, X_train):
        self.model.fit(X_train)

        print("Gaussian Mixture Model Training")
        print("--------------------------------")
        print("Converged :", self.model.converged_)
        print("Iterations:", self.model.n_iter_)

    def evaluate(self, X_test):
        score = self.model.score(X_test)

        print("\nValidation Performance")
        print("----------------------")
        print(f"Average Log-Likelihood : {score:.4f}")

        return score

    def density_heatmap(self, X_train, feature_names):
        original = self.scaler.inverse_transform(X_train)

        fig = px.density_heatmap(
            x=original[:, 0],
            y=original[:, 1],
            labels={
                "x": feature_names[0],
                "y": feature_names[1]
            },
            marginal_x="histogram",
            marginal_y="histogram",
            title="Empirical Density of Training Data"
        )

        fig.update_traces(
            colorscale="Viridis",
            selector=dict(type="histogram2d")
        )

        fig.update_layout(template="plotly_white")

        fig.show()

    def contour_surface(self, X):

        xmin = X[:, 0].min() - 0.5
        xmax = X[:, 0].max() + 0.5

        ymin = X[:, 1].min() - 0.5
        ymax = X[:, 1].max() + 0.5

        xx, yy = np.meshgrid(
            np.linspace(xmin, xmax, 250),
            np.linspace(ymin, ymax, 250)
        )

        grid = np.c_[xx.ravel(), yy.ravel()]

        responsibilities = self.model.predict_proba(grid)

        posterior = responsibilities.max(axis=1).reshape(xx.shape)

        original = self.scaler.inverse_transform(grid)

        xx_original = original[:, 0].reshape(xx.shape)
        yy_original = original[:, 1].reshape(yy.shape)

        return xx_original, yy_original, posterior

    def plot_training_assignments(self, X_train, feature_names):

        xx, yy, posterior = self.contour_surface(X_train)

        labels = self.model.predict(X_train)

        train_original = self.scaler.inverse_transform(X_train)

        fig = go.Figure()

        fig.add_trace(
            go.Contour(
                x=xx[0],
                y=yy[:, 0],
                z=posterior,
                colorscale="Cividis",
                contours_coloring="heatmap",
                opacity=0.6,
                showscale=True,
                name="Posterior Probability"
            )
        )

        for k in range(self.n_components):

            mask = labels == k

            fig.add_trace(
                go.Scatter(
                    x=train_original[mask, 0],
                    y=train_original[mask, 1],
                    mode="markers",
                    name=f"Cluster {k+1}",
                    marker=dict(size=6)
                )
            )

        fig.update_layout(
            title="Training Data with Posterior Responsibility Contours",
            xaxis_title=feature_names[0],
            yaxis_title=feature_names[1],
            template="plotly_white"
        )

        fig.show()

    def plot_test_assignments(self, X_test, feature_names):

        xx, yy, posterior = self.contour_surface(X_test)

        labels = self.model.predict(X_test)

        test_original = self.scaler.inverse_transform(X_test)

        fig = go.Figure()

        fig.add_trace(
            go.Contour(
                x=xx[0],
                y=yy[:, 0],
                z=posterior,
                colorscale="Cividis",
                contours_coloring="heatmap",
                opacity=0.6,
                showscale=True,
                name="Posterior Probability"
            )
        )

        for k in range(self.n_components):

            mask = labels == k

            fig.add_trace(
                go.Scatter(
                    x=test_original[mask, 0],
                    y=test_original[mask, 1],
                    mode="markers",
                    name=f"Cluster {k+1}",
                    marker=dict(size=6)
                )
            )

        fig.update_layout(
            title="Out-of-Sample Test Data with Posterior Responsibility Contours",
            xaxis_title=feature_names[0],
            yaxis_title=feature_names[1],
            template="plotly_white"
        )

        fig.show()


features = [
    "PURCHASES",
    "CREDIT_LIMIT"
]

segmenter = GMMFinancialSegmenter(
    n_components=3,
    random_state=42
)

X_train, X_test = segmenter.prepare_data(
    df,
    features
)

segmenter.fit(X_train)

segmenter.evaluate(X_test)

segmenter.density_heatmap(
    X_train,
    features
)

segmenter.plot_training_assignments(
    X_train,
    features
)

segmenter.plot_test_assignments(
    X_test,
    features
)

print("\nInterpretation")
print("--------------")
print("The density heatmap illustrates the empirical distribution of the financial data and highlights the presence of multiple dense regions that motivate the use of a Gaussian Mixture Model.")
print("The contour plots display the maximum posterior responsibility across the feature space. These continuous contours visualize the soft assignment vector E[Z|X=x], where every location is associated with a probability of belonging to each Gaussian component.")
print("Regions with smooth colour transitions indicate uncertainty and overlapping Gaussian components, while regions with uniform colouring indicate high-confidence cluster assignments.")
print("The training plot confirms that the EM algorithm successfully partitions the observed data into three probabilistic clusters.")
print("The test plot demonstrates that the learned Gaussian density functions generalize well to unseen observations, with most validation samples falling inside regions of high posterior confidence.")
print("Overall, the visualizations verify the theoretical result that Gaussian Mixture Models perform probabilistic clustering through conditional expectations of latent cluster membership variables rather than deterministic assignments.")


# Gaussian Mixture Models as Conditional Expectation Models

## Part 1 – Marginal Density and Posterior Cluster Probabilities

# ---

# 1. Deriving the Marginal Density

Consider a dataset

$$
x_1,x_2,\dots,x_n\in\mathbb{R}^d.
$$

A Gaussian Mixture Model (GMM) assumes that every observation is generated from one of $K$ latent Gaussian components. The hidden cluster membership of observation $x_i$ is represented by the random variable

$$
C_i\in\{1,2,\ldots,K\},
$$

with prior probability

$$
P(C_i=k)=\phi_k,
$$

where

$$
\phi_k\ge0,
\qquad
\sum_{k=1}^{K}\phi_k=1.
$$

Conditional on belonging to cluster $k$, the observation follows a multivariate Gaussian distribution,

$$
X_i\mid C_i=k
\sim
\mathcal{N}(\mu_k,\Sigma_k).
$$

To determine the unconditional (marginal) density of an observation, we apply the **Law of Total Probability** by summing over every possible cluster assignment.

The joint probability of observing $x_i$ and belonging to cluster $k$ is

$$
P(X_i=x_i,C_i=k)
=
P(X_i=x_i\mid C_i=k)\,P(C_i=k).
$$

Therefore,

$$
p(x_i)
=
\sum_{k=1}^{K}
P(X_i=x_i,C_i=k).
$$

Substituting the Gaussian likelihood and the prior mixture probabilities gives

$$
p(x_i)
=
\sum_{k=1}^{K}
P(X_i=x_i\mid C_i=k)\,P(C_i=k)
=
\sum_{k=1}^{K}
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k).
$$

Hence, the marginal density of an observation is

$$
\boxed{
p(x_i)
=
\sum_{k=1}^{K}
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}
$$

# ---

### Why is this called a Gaussian Mixture Density?

The resulting probability density is called a **Gaussian mixture density** because it is constructed as a weighted combination of several Gaussian component distributions.

Each Gaussian distribution

$$
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
$$

models one underlying cluster in the dataset, while the mixture weight

$$
\phi_k
$$

represents the prior probability that an arbitrary observation originates from cluster $k$.

The complete density therefore combines multiple Gaussian distributions into a single probability model,

$$
p(x_i)
=
\phi_1\mathcal{N}_1
+
\phi_2\mathcal{N}_2
+\cdots+
\phi_K\mathcal{N}_K.
$$

Unlike a single Gaussian distribution, which can only represent one approximately elliptical cluster, a Gaussian mixture density can approximate highly irregular, skewed, or multimodal distributions by combining several Gaussian components together.

# ---

# 2. Deriving the Posterior Cluster Probability

After observing a particular data point $x_i$, we wish to determine the probability that it originated from each cluster.

Using **Bayes' Theorem**,

$$
P(C_i=k\mid X_i=x_i)
=
\frac{
P(X_i=x_i\mid C_i=k)\,P(C_i=k)
}{
P(X_i=x_i)
}.
$$

From Part 1,

$$
P(X_i=x_i)
=
\sum_{j=1}^{K}
\phi_j
\mathcal{N}(x_i\mid\mu_j,\Sigma_j).
$$

Substituting the likelihood and prior gives

$$
P(C_i=k\mid X_i=x_i)
=
\frac{
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}{
\sum_{j=1}^{K}
\phi_j
\mathcal{N}(x_i\mid\mu_j,\Sigma_j)
}.
$$

The posterior probability is therefore

$$
\boxed{
P(C_i=k\mid X_i=x_i)
=
\frac{
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}{
\sum_{j=1}^{K}
\phi_j
\mathcal{N}(x_i\mid\mu_j,\Sigma_j)
}
}
$$

This quantity is called the **responsibility** of cluster $k$ for observation $x_i$ and is denoted by

$$
\boxed{
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i)
}.
$$

# ---

### Interpretation of the Responsibility

The responsibility

$$
\gamma_{ik}
$$

is the **posterior probability** that observation $x_i$ belongs to cluster $k$ after incorporating the observed data.

Its interpretation follows directly from Bayes' theorem:

- **Prior Information:** Before observing the data, the probability of belonging to cluster $k$ is represented by the mixture weight

  $$
  \phi_k.
  $$

- **Likelihood:** The Gaussian density

  $$
  \mathcal{N}(x_i\mid\mu_k,\Sigma_k)
  $$

  measures how compatible the observed point is with cluster $k$.

- **Posterior Update:** Bayes' theorem combines these two sources of information to produce the updated probability

  $$
  \gamma_{ik}.
  $$

The responsibilities satisfy

$$
0\le\gamma_{ik}\le1,
$$

and

$$
\sum_{k=1}^{K}\gamma_{ik}=1,
$$

which means every observation distributes one unit of probability across all clusters.

Observations located near the overlap of two Gaussian components may have responsibilities such as

$$
(\gamma_{i1},\gamma_{i2},\gamma_{i3})
=
(0.45,\;0.50,\;0.05),
$$

indicating uncertainty in cluster membership.

Conversely, a point lying very close to the center of one cluster may have responsibilities such as

$$
(0.98,\;0.01,\;0.01),
$$

showing that nearly all posterior probability is assigned to a single cluster.

Thus, the responsibility provides a **soft probabilistic assignment** instead of making an immediate hard decision about cluster membership.




# Part 2

---

# 4. From Soft Assignment to Hard Clustering

In a Gaussian Mixture Model (GMM), the conditional expectation vector

$$
\mathbb{E}[Z_i \mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}
$$

provides a **soft assignment** of observation $x_i$ across all clusters.

Each responsibility satisfies

$$
0 \le \gamma_{ik}\le 1,
\qquad
\sum_{k=1}^{K}\gamma_{ik}=1,
$$

meaning every observation belongs to every cluster with different posterior
probabilities.

The corresponding **hard assignment** is obtained by selecting the cluster with
the highest posterior probability:

$$
\widehat C_i
=
\operatorname*{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$

---

### Soft Clustering

Soft clustering preserves uncertainty in the cluster assignment.

Instead of forcing a point into a single cluster, the model produces an entire
probability vector describing how strongly the observation belongs to every
component.

For example,

$$
\mathbb{E}[Z_i\mid X_i=x_i]
=
\begin{bmatrix}
0.15\\
0.70\\
0.15
\end{bmatrix}
$$

indicates that the observation most likely belongs to Cluster 2 while still
retaining some probability of belonging to Clusters 1 and 3.

This probabilistic representation is especially useful when observations lie
near overlapping cluster boundaries.

---

### Hard Clustering

Hard clustering converts the posterior probability vector into a single class
label.

Using the previous example,

$$
\widehat C_i
=
\operatorname*{arg\,max}
(0.15,\;0.70,\;0.15)
=
2.
$$

The observation is assigned entirely to Cluster 2, while the remaining
probabilities are discarded.

---

### Comparison

| Soft Clustering | Hard Clustering |
|----------------- |-----------------|
| Produces posterior probabilities | Produces one cluster label |
| Preserves uncertainty | Removes uncertainty |
| Observation belongs partially to every cluster | Observation belongs to exactly one cluster |
| Used during EM parameter estimation | Used for final classification and visualization |

---

### Conclusion

The conditional expectation

$$
\mathbb{E}[Z_i\mid X_i=x_i]
$$

contains the complete probabilistic description of cluster membership.

The hard assignment

$$
\widehat C_i
=
\operatorname*{arg\,max}_{k}
\gamma_{ik}
$$

is simply a deterministic decision obtained from this probability vector.

---

# 5. Conditional Expectation of the Observation Given the Cluster

By definition of the Gaussian Mixture Model,

$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k).
$$

The expected value of a multivariate Gaussian distribution equals its mean
vector.

Therefore,

$$
\boxed{
\mathbb E[X_i\mid C_i=k]=\mu_k
}
$$

or equivalently,

$$
\mathbb E[X_i\mid C_i=k]
=
\int_{\mathbb R^d}
x\,
\mathscr N(x\mid\mu_k,\Sigma_k)
dx
=
\mu_k.
$$

---

## Why is $\mu_k$ the Center of Cluster $k$?

The vector $\mu_k$ represents the centroid of the Gaussian component because

* it is the expected location of observations generated from cluster $k$,

* it is the point where the Gaussian density reaches its maximum,

* the Gaussian distribution is symmetric around $\mu_k$,

* every direction away from $\mu_k$ balances equally in expectation.

Hence,

$$
\boxed{
\mu_k
=
\text{Center of Cluster }k
}
$$

---

## Comparing the Two Conditional Expectations

### (a)

$$
\mathbb E[Z_i\mid X_i=x_i]
$$

This quantity conditions on an observed data point.

The location of the point is fixed, while the cluster identity is unknown.

Its output is

$$
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix},
$$

which represents the posterior probability that the observation belongs to each
cluster.

Therefore,

$$
\boxed{
\mathbb E[Z_i\mid X_i=x_i]
=
\text{Soft cluster membership vector.}
}
$$

---

### (b)

$$
\mathbb E[X_i\mid C_i=k]
$$

This quantity conditions on the cluster identity.

The cluster is fixed, while the observation is random.

Its output is

$$
\mu_k,
$$

the average location of observations generated from that cluster.

Therefore,

$$
\boxed{
\mathbb E[X_i\mid C_i=k]
=
\text{Center of Cluster }k.
}
$$

---

### Summary Comparison

| Quantity | Conditions On | Unknown Variable | Output | Interpretation |
|-----------|---------------|-----------------|--------|----------------|
| $\mathbb E[Z_i\mid X_i=x_i]$ | Observation | Cluster identity | Probability vector | Soft cluster membership |
| $\mathbb E[X_i\mid C_i=k]$ | Cluster | Observation | Mean vector | Cluster center |

---

### Conclusion

Although both expressions are conditional expectations, they answer completely
different questions.

The expectation

$$
\mathbb E[Z_i\mid X_i=x_i]
$$

estimates **which cluster generated an observed point**, whereas

$$
\mathbb E[X_i\mid C_i=k]
$$

estimates **where observations from a particular cluster are expected to be
located**.

---

# 6. The Complete-Data Likelihood

Suppose that the latent indicator variables

$$
z_{ik}
\in
\{0,1\}
$$

are observed.

The complete-data likelihood is

$$
p(x_1,\ldots,x_n,z_1,\ldots,z_n)
=
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathscr N(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$

---

## Taking the Logarithm

Applying the logarithm,

$$
\ell_c
=
\log
\left(
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathscr N(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}
\right).
$$

Using

$$
\log\left(\prod a_i\right)
=
\sum\log(a_i),
$$

gives

$$
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\log
\left(
\left[
\phi_k
\mathscr N(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}
\right).
$$

Applying

$$
\log(a^b)=b\log(a),
$$

yields

$$
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\log
\left(
\phi_k
\mathscr N(x_i\mid\mu_k,\Sigma_k)
\right).
$$

Finally, using

$$
\log(ab)
=
\log a+\log b,
$$

we obtain

$$
\boxed{
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
+
\log
\mathscr N(x_i\mid\mu_k,\Sigma_k)
\right]
}
$$

which is the required complete-data log-likelihood.

---

## Why is this Easy to Maximize?

If every indicator variable $z_{ik}$ were known, then the assignment of every
observation to a cluster would already be available.

Consequently,

* each Gaussian component could be estimated independently,

* there would be no logarithm of a summation,

* observations assigned to each cluster could be treated as belonging to a
single Gaussian distribution,

* the mixture weights would simply equal the proportion of observations assigned
to each cluster.

The maximum likelihood estimates become

$$
\phi_k
=
\frac{\text{Number of observations in Cluster }k}{n},
$$

$$
\mu_k
=
\text{Sample mean of Cluster }k,
$$

and

$$
\Sigma_k
=
\text{Sample covariance of Cluster }k.
$$

---

### Conclusion

The complete-data log-likelihood is straightforward to optimize because the
hidden variables separate the observations into independent Gaussian
components.

The difficulty of Gaussian Mixture Models arises precisely because these latent
variables are unknown, motivating the Expectation-Maximization (EM) algorithm,
which replaces the unknown indicators with their conditional expectations.





# Part 6. The Complete-Data Likelihood

## 1. Derivation of the Complete-Data Log-Likelihood

If the latent cluster indicators are observed, the complete-data likelihood is

$$
p(x_1,\ldots,x_n,z_1,\ldots,z_n)
=
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$

Taking the natural logarithm,

$$
\ell_c
=
\log
\left(
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}
\right).
$$

Using the logarithm property

$$
\log\left(\prod a_i\right)=\sum\log(a_i),
$$

gives

$$
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\log
\left(
\left[
\phi_k
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}
\right).
$$

Applying

$$
\log(a^b)=b\log(a),
$$

we obtain

$$
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\log
\left(
\phi_k
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right).
$$

Finally, using

$$
\log(ab)=\log a+\log b,
$$

the complete-data log-likelihood becomes

$$
\boxed{
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
+
\log
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right]
}
$$

which is the required expression.

---

## 2. Why the Complete-Data Log-Likelihood is Easy to Maximize

If every latent indicator $z_{ik}$ were known, the optimization problem would become much simpler.

Each observation would already be assigned to a particular Gaussian component, so there would be no uncertainty regarding cluster membership.

The logarithm transforms the product of probabilities into a summation, causing the contribution of every Gaussian component to become additive rather than coupled through a logarithm of sums.

Consequently,

* the mixture weights $\phi_k$ can be estimated directly from the proportion of observations assigned to each cluster,

* the mean vector $\mu_k$ becomes the sample average of the observations assigned to cluster $k$, and

* the covariance matrix $\Sigma_k$ becomes the sample covariance of those observations.

Thus, if the values of $z_{ik}$ were known, the difficult mixture optimization problem decomposes into $K$ independent maximum-likelihood estimation problems for ordinary multivariate Gaussian distributions.

---

# Part 7. The EM Interpretation

## 1. Expected Complete-Data Log-Likelihood

In practice, the latent indicators $z_{ik}$ are never observed.

Instead, the Expectation-Maximization (EM) algorithm replaces every unknown indicator by its conditional expectation given the observed data and the current parameter estimates.

Beginning with the complete-data log-likelihood,

$$
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
+
\log
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right],
$$

we take the conditional expectation with respect to the posterior distribution of the latent variables:

$$
Q
=
\mathbb{E}
\left[
\ell_c
\mid
X=x
\right].
$$

Substituting the expression for $\ell_c$,

$$
Q
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\mathbb{E}
[
Z_{ik}
\mid
X_i=x_i
]
\left[
\log\phi_k
+
\log
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right].
$$

From Part 3,

$$
\mathbb{E}
[
Z_{ik}
\mid
X_i=x_i
]
=
\gamma_{ik},
$$

where

$$
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i)
$$

is the posterior responsibility.

Therefore,

$$
\boxed{
Q
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\gamma_{ik}
\left[
\log\phi_k
+
\log
\mathscr{N}(x_i\mid\mu_k,\Sigma_k)
\right]
}
$$

which is called the expected complete-data log-likelihood or the Q-function.

---

## 2. Interpretation of the E-Step

The Expectation step performs a conditional update of the latent cluster memberships.

Instead of assigning each observation permanently to one cluster, the algorithm computes the posterior probability that every observation belongs to every Gaussian component.

Each responsibility

$$
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i)
$$

combines

* the prior probability of cluster $k$,

$$
\phi_k,
$$

and

* the likelihood that cluster $k$ generated the observation,

$$
\mathscr{N}(x_i\mid\mu_k,\Sigma_k),
$$

through Bayes' theorem.

Consequently, the E-step transforms the unknown binary indicator variables into continuous probabilities representing updated beliefs about cluster membership.

These posterior probabilities become the weights used during the Maximization step to update the model parameters.

---

# Part 8. Parameter Updates

After computing the posterior responsibilities during the E-step, the Maximization step updates every model parameter by maximizing the expected complete-data log-likelihood.

## Effective Number of Observations

The effective number of observations assigned to cluster $k$ is

$$
\boxed{
N_k
=
\sum_{i=1}^{n}
\gamma_{ik}
}
$$

which represents the total posterior membership weight of cluster $k$.

---

## Updating the Mixture Weights

The updated prior probability of cluster $k$ is

$$
\boxed{
\phi_k^{\mathrm{new}}
=
\frac{N_k}{n}
}
$$

ensuring

$$
\sum_{k=1}^{K}
\phi_k^{\mathrm{new}}
=
1.
$$

---

## Updating the Cluster Means

The updated mean vector is

$$
\boxed{
\mu_k^{\mathrm{new}}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}x_i
}
$$

which is a weighted average of all observations.

Observations with larger posterior probabilities contribute more heavily to the cluster center.

---

## Updating the Covariance Matrices

The updated covariance matrix is

$$
\boxed{
\Sigma_k^{\mathrm{new}}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}
(x_i-\mu_k^{\mathrm{new}})
(x_i-\mu_k^{\mathrm{new}})^T
}
$$

which measures the weighted spread of the observations around the updated mean.

---

## Interpretation of the Responsibilities

The responsibility

$$
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i)
$$

acts as a fractional membership weight rather than a binary assignment.

If

$$
\gamma_{ik}=1,
$$

the observation belongs completely to cluster $k$.

If

$$
\gamma_{ik}=0,
$$

the observation contributes nothing to that cluster.

Intermediate values indicate partial membership, allowing observations located between overlapping Gaussian components to influence multiple clusters simultaneously.

This weighted updating mechanism enables the Gaussian Mixture Model to represent uncertainty in cluster membership while continuously refining the model parameters throughout the EM iterations.



Using Colab cache for faster access to the 'ccdata' dataset.
Dataset Path: /kaggle/input/ccdata
Files:
['CC GENERAL.csv']
Gaussian Mixture Model Training
--------------------------------
Converged : True
Iterations: 19

Validation Performance
----------------------
Average Log-Likelihood : -1.6465



Interpretation
--------------
The density heatmap illustrates the empirical distribution of the financial data and highlights the presence of multiple dense regions that motivate the use of a Gaussian Mixture Model.
The contour plots display the maximum posterior responsibility across the feature space. These continuous contours visualize the soft assignment vector E[Z|X=x], where every location is associated with a probability of belonging to each Gaussian component.
Regions with smooth colour transitions indicate uncertainty and overlapping Gaussian components, while regions with uniform colouring indicate high-confidence cluster assignments.
The training plot confirms that the EM algorithm successfully partitions the observed data into three probabilistic clusters.
The test plot demonstrates that the learned Gaussian density functions generalize well to unseen observations, with most validation samples falling inside regions of high posterior confidence.
Overall, the visualizations